# NB-12 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-12**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-12.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This executes the volume’s separate vintage and European arithmetic examples. It shows how a forecast revision, forecast error, and derived market field receive different identities and cutoffs.

- [Volume V, Chapter 6 — Valid time, knowledge time, and revisions](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c6)
- [Volume V, Chapter 11 — European power, gas, carbon, and weather](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c11)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-12', require_key=True)
print(setup_status)


# NB-12 — Forecast vintages, surprise and the European trace
Companion to Volume V §§5–8, 11 and 14.2. Prerequisites: NB-00 and NB-11.
Reuse the book's existing A/B/C vintage example and its separate European
arithmetic trace. Values are parsed from the manuscript, not newly simulated.
No historical API reproduction or empirical forecasting claim is made.

The examples cover different delivery hours and must not be joined. Chapter
6.7 is the 18:00–19:00 vintage timeline; Chapter 11.5 is the 17:00–18:00 European
trace. We interpret the chapter's clock examples in UTC and keep missing run
and publication timestamps null.

In [ ]:
NOTEBOOK_ID='NB-12'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

rows=inputs['vintages'];cap={'generation':inputs['api_gate']};cutoffs=inputs['cutoffs']
rejected_controls=set();private_outputs={}
def invoke(name,parents=(),request=None):
    entry=inputs['requests'][name]
    output=execute(name,entry['method'],entry['request'] if request is None else request,parents)
    private_outputs[name]=output
    return output
def refuse(name):
    control=next(c for c in inputs['negative_controls'] if c['name']==name)
    reject(control['method'],control['request']);rejected_controls.add(name)
arithmetic=invoke('retained-european-arithmetic')['result']['illustrative_values'];trace=arithmetic['trace']
for row in rows:print(row)
assert all(r['published'] is None and r['run'] is None for r in rows)
print('API extension:',cap['generation'])

## Five clocks answer different questions
Delivery time defines the target interval. Issue time orders the source's
forecasts; publication time is the release clock; receipt time records
collection. Workbench snapshot time identifies the captured input version.
It is not an assertion's historical receipt time.

The retained manuscript supplies issue and receipt times for A/B/C, but no
publication or run time. Collector-as-of can therefore be demonstrated under
that contract. Source-published-as-of must fail explicitly. The local input
receipt pins the manuscript bytes; it does not create missing source clocks.

In [ ]:
audit=[];selection_outputs=[]
latest=invoke('latest_retained_hindsight')['result']['selected_assertions']['records'][0]
for i,cutoff in enumerate(cutoffs):
    selected=invoke('cutoff:'+str(i))['result']['selected_assertions'];selection_outputs.append(selected)
    chosen=selected['records'][0]
    audit.append(dict(cutoff=cutoff,collector_assertion=chosen['assertion_id'],collector_MW=chosen['value'],latest_assertion=latest['assertion_id'],latest_MW=latest['value']))
    print(audit[-1])
assert [r['collector_assertion'] for r in audit]==['A','B','C']
refuse('missing_publication')
boundary_results=[]
for i,row in enumerate(rows):
    at_receipt=invoke(row['assertion']+':receipt')['result']['selected_assertions']['records']
    at_issue=invoke(row['assertion']+':issue')['result']['selected_assertions']['records']
    assert [r['assertion_id'] for r in at_receipt]==[row['assertion']]
    assert [r['assertion_id'] for r in at_issue]==([] if i==0 else [rows[i-1]['assertion']])
    boundary_results.append(dict(assertion=row['assertion'],at_receipt=at_receipt,at_issue=at_issue))
fig,ax=plt.subplots(figsize=(9,4));x=np.arange(3)
ax.plot(x,[r['collector_MW'] for r in audit],'o-',label='Collector-as-of',color='#88C0D0')
ax.plot(x,[r['latest_MW'] for r in audit],'s--',label='Latest retained assertion · hindsight',color='#D08770')
ax.set_xticks(x,['06:00','09:00','16:00']);ax.set(xlabel='Decision cutoff on 2026-02-12 UTC',ylabel='Forecast MW',title='One delivery hour · cutoff changes the admissible assertion');ax.legend();fig.tight_layout();plt.show()

## A one-interval vintage triangle, with future cells absent
Each row below is a decision cutoff and each column an assertion. Blank cells
mean not yet received. There is no interpolation, backdating, or replacement
of an earlier forecast by its later correction. This is one retained interval,
not a fabricated multi-day panel.

In [ ]:
# Display source values only where the private selector's eligibility ledger permits.
matrix=np.array([[next(r['value'] for r in rows if r['assertion']==entry['assertion_id']) if entry['eligible'] else np.nan for entry in selected['selection_ledger']] for selected in selection_outputs])
from matplotlib.colors import LinearSegmentedColormap
nord=LinearSegmentedColormap.from_list('nord_vintage',['#5E81AC','#88C0D0','#ECEFF4']);nord.set_bad('#3B4252')
fig,ax=plt.subplots(figsize=(7,4));im=ax.imshow(matrix,cmap=nord,aspect='auto')
for i in range(3):
    for j in range(3):ax.text(j,i,'not yet known' if np.isnan(matrix[i,j]) else f'{matrix[i,j]:,.0f}',ha='center',va='center',color='#ECEFF4' if np.isnan(matrix[i,j]) else '#2E3440')
ax.set_xticks(range(3),['A','B','C']);ax.set_yticks(range(3),['06:00','09:00','16:00']);ax.set(xlabel='Source assertion',ylabel='Collector cutoff UTC',title='Admissible vintages · same delivery interval');fig.colorbar(im,ax=ax,label='MW');fig.tight_layout();plt.show()

## Revision is not realized forecast error
In the Chapter 6 timeline, B−A and C−B become known only when B and C arrive.
That timeline has no realization, so its errors are unavailable.
The separate Chapter 11 trace supplies both prior/current forecasts and a
later realization: its revision is +1,500 MW and error is −1,200 MW.
The realization belongs to evaluation cutoff T3, never forecast decision T1.

In [ ]:
revision_rows=[]
for metadata in inputs['revision_metadata']:
    output=invoke('revision:'+metadata['to'])['result']['forecast_difference']
    difference=output['rows'][0]
    assert (difference['forecast_id'],difference['later_id'],difference['unit'])==(metadata['from'],metadata['to'],metadata['unit'])
    assert output['cutoff']==metadata['known_at']
    revision_rows.append(dict(metadata,value=difference['difference']))

for revision in revision_rows:print(revision)
print('Separate Chapter 11 trace:',{k:trace[k] for k in ('revision_MW','forecast_error_MW','residual_load_MW')})
np.testing.assert_allclose([trace['revision_MW'],trace['forecast_error_MW'],trace['residual_load_MW']],[1500,-1200,43300])
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].bar(['B − A','C − B'],[r['value'] for r in revision_rows],color=['#D08770','#88C0D0']);axes[0].set(ylabel='MW',title='Chapter 6 · adjacent revisions')
axes[1].bar(['Revision at T1','Error at T3'],[trace['revision_MW'],trace['forecast_error_MW']],color=['#88C0D0','#D08770']);axes[1].set(ylabel='MW',title='Chapter 11 · different objects and clocks');fig.tight_layout();plt.show()

## Surprise normalization: insufficient history is a result
The retained registry contract requires 180 days of trailing history and at
least 60 observations in a compatible bucket. Our example supplies only two
revisions. It cannot establish a lead/hour/season reference distribution.
A standardized score is therefore null with an explicit reason. We do not
reuse future revisions, pool incompatible buckets, or invent a reference sample.
The helper excludes the current revision and any later-known assertion.

In [ ]:
surprises=[]
for revision in revision_rows:
    name='surprise:'+revision['to'];request=copy.deepcopy(inputs['requests'][name]['request'])
    request['inputs']['train_eval_panel'].update(training_pool=revision_rows,evaluation=revision)
    output=invoke(name,['revision:B','revision:C'],request)['result']
    scored=output['standardized_panel']['evaluation'];surprises.append(scored);print(revision['to'],scored)
    assert output['fit_parameter_summary']['fitted'] is False
assert all(s['score'] is None for s in surprises)
assert [s['n'] for s in surprises]==[0,1]

## European arithmetic: units, provider identity and contract boundaries
Chapter 11's illustrative T2 clean-spark arithmetic uses power and gas in
EUR/MWh, carbon in EUR/tCO2, heat rate 2.0 and emissions factor 0.411.
It gives −0.0535 EUR/MWh, displayed in the book as −0.05. This is an archetype
calculation, not a validated tradable spread: actual tenor, roll and settlement
matching are not supplied by the illustrative trace.

N2EX and APX remain separate MIP provider targets. The retained selection
policy chooses N2EX only in the T3 evaluation surface. Multiplication by
EUR_per_GBP gives a selected normalized MIP; it does not create a GB day-ahead
auction price. FX admissibility is cutoff-dependent, not a blanket T1 input.

In [ ]:
for key in ('power_EUR_MWh','gas_EUR_MWh','carbon_EUR_tCO2','heat_rate','emissions_factor','clean_spark_EUR_MWh','N2EX_MIP_GBP_MWh','APX_MIP_GBP_MWh','fx_EUR_per_GBP','selected_MIP_EUR_MWh'):print(key,trace[key])
np.testing.assert_allclose(trace['clean_spark_EUR_MWh'],-.0535,atol=1e-10)
np.testing.assert_allclose(trace['selected_MIP_EUR_MWh'],101.7792,atol=1e-10)
fig,ax=plt.subplots(figsize=(9,4))
parts=arithmetic['plot_components']
ax.bar(['Power','Gas cost','Carbon cost','Clean spark'],parts,color=['#88C0D0','#D08770','#B48EAD','#A3BE8C']);ax.axhline(0,color='#ECEFF4',lw=.7)
ax.set(ylabel='EUR/MWh electricity',title='Retained Chapter 11 arithmetic · contract validation not supplied');fig.tight_layout();plt.show()

## Exercise and remaining gates
Explain why the latest forecast is inadmissible at 06:00, why issue time cannot
substitute for publication time, and why a null surprise score is preferable
to a normalized number from two observations.
Real-data extension requires a working descriptor, verified vintage coverage,
actual publication/receipt semantics, and compatible realization and reference
channels. The current catalog/descriptor conflict remains unresolved. Neither
the book arithmetic nor the prototype registry closes those gaps.

In [ ]:
for control in inputs['negative_controls']:
    if control['name'] not in rejected_controls:refuse(control['name'])
checked=dict(vintages=rows,selection_audit=audit,revisions=revision_rows,surprise_status=surprises,european_trace=trace,api_gate=cap['generation'])
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(existing_vintages_parsed=True,collector_cutoffs=True,published_asof_rejected_without_clock=True,revision_error_distinct=True,insufficient_surprise_support_explicit=True,fx_multiplication=True,provider_targets_preserved=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)